# SCFM Example Notebook
**Supernova Classification with FPCA and Machine Learning**

This notebook demonstrates the full SCFM pipeline:
1. Fitting light curves using FPCA
2. Inspecting FPCA fit results
3. Identifying potential SNe Ia

## 1. Installation

Install SCFM via pip:

```bash
pip install scfm
```

If you want to modify the source code, install in editable mode instead:

```bash
git clone https://github.com/MoonzarinReza/scfm.git
cd scfm
pip install -e .
```

In [ ]:
# Uncomment only for local development before pip install
# import os
# os.chdir('/Volumes/new_drive/Desirt_Work/Desirt_Work_Test/fail_again/Mapping/Github_Upload/abc')
# !pip install --upgrade setuptools
# !pip install -e. 

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import scfm
from scfm import fit_lc, classify

## 2. Data

Each light curve must be stored as a separate CSV file inside a directory. The CSV file must contain the following columns:

| Column | Description |
|--------|-------------|
| `mjd` | Modified Julian Date of the observation |
| `mag` | Apparent magnitude |
| `magerr` | Magnitude uncertainty |
| `filter` | Photometric band (e.g. `g`, `r`, `i`, `z`, `y`) |
| `redshift` | Spectroscopic or photometric redshift of the transient |
| `name` | Unique identifier for the transient |

**Note:** If all your light curves are stored in a single CSV file, you can split them into individual files by transient name as follows:

In [ ]:
# Optional: split a single CSV into per-transient files
# df = pd.read_csv('all_lightcurves.csv')
# os.makedirs('sample_lightcurves', exist_ok=True)
# for name, group in df.groupby('name'):
#     group.to_csv(f'sample_lightcurves/{name}.csv', index=False)

## 3. Fit Light Curves

`fit_lc` performs FPCA-based light curve fitting using Levenberg-Marquardt optimization. For each transient and each photometric band, it fits the model:

$$g(q) = m + \phi_0(q) + a_1 \phi_1(q) + a_2 \phi_2(q)$$

where $m$ is a fitted parameter representing the apparent peak magnitude, $\phi_0$, $\phi_1$, $\phi_2$ are the fixed mean template and the first two principal eigenvectors learned from a training set of SNe Ia light curves, and $a_1$, $a_2$ are the FPCA scores that characterize each individual light curve.

### Parameters

| Parameter | Description | Default |
|-----------|-------------|---------|
| `lc_dir` | Root directory containing the light curve CSV files. Each CSV file corresponds to one transient. | `'sample_lightcurves'` |
| `num_attempts` | Number of times the best-fit parameters are determined; the optimum solution is selected based on the minimum chi-square value. | `3` |
| `min_points` | Minimum number of observations per filter required for fitting. | `5` |

In [ ]:

sample_lc_dir = os.path.join(os.path.dirname(scfm.__file__), 'sample_lightcurves')

df_final = fit_lc(lc_dir=sample_lc_dir, num_attempts=3, min_points=5)

### Outputs

| File | Description |
|------|-------------|
| `raw.csv` | Per-filter fit results (one row per filter per transient) |
| `wide.csv` | Wide format with one row per transient |
| `final_fpca_data.csv` | Same as `wide.csv` but with apparent magnitudes replaced by absolute magnitudes; ready for classification |
| `plots/` | Fitted light curve plots for each transient |
| `tables/` | Per-filter photometry tables |

## 4. Inspect FPCA Fit Results

In [ ]:
# Load and inspect the fit results
df_final = pd.read_csv('final_fpca_data.csv')
print(f'Number of successfully fitted transients: {len(df_final)}')
print(df_final.head())

In [ ]:
# Show a few fitted light curve plots
plot_files = [f for f in os.listdir('plots') if f.endswith('.png')]
print(f'Total fitted light curves: {len(plot_files)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, pfile in zip(axes, plot_files[:3]):
    img = plt.imread(os.path.join('plots', pfile))
    ax.imshow(img)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Classify

`classify` uses CatBoost  as the base classifier within a majority-vote ensemble (n_runs, min_votes, and threshold are adjustable). Features used for classification are the FPCA scores in the available photometric bands and m converted to absolute magnitudes assuming a fixed cosmology.

### Parameters

| Parameter | Description | Default |
|-----------|-------------|---------|
| `n_runs` | Number of times the experiment is repeated. | `5` |
| `min_votes` | Minimum number of runs that must predict SNe Ia for a transient to be accepted as a candidate. | `3` |
| `threshold` | Predicted probability above which a transient is determined as SNe Ia in a given run. Must be between 0 and 1. | `0.5` |

In [ ]:
df_candidates=classify(
    n_runs = 5,
    min_votes = 3,
    threshold = 0.5
)

### Output

| File | Description |
|------|-------------|
| `potential_SNIa_candidates.csv` | Names and redshifts of SNe Ia candidates |

## 6. Results

In [ ]:
# Load classification results
df_candidates = pd.read_csv('potential_SNIa_candidates.csv')
print(f'Total SNe Ia candidates identified: {len(df_candidates)}')
print(df_candidates.head(10))

In [ ]:
# Redshift distribution of SNe Ia candidates
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df_candidates['redshift'], bins=20, color='dodgerblue', edgecolor='white')
ax.set_xlabel('Redshift')
ax.set_ylabel('Count')
ax.set_title('Redshift Distribution of SNe Ia Candidates')
plt.tight_layout()
plt.show()

## Citation

If you use SCFM in your research, please cite:

> Reza, M., Wang, L., & Hu, L. (2025). An FPCA-Enhanced Ensemble Learning Framework for Photometric Identification of Type Ia Supernovae. *arXiv:2510.09990*.